In [0]:
# Import ml flow library to auto-log machine learning runs

import mlflow

mlflow.pyspark.ml.autolog()

# CREATING DATFRAME

In [0]:
# improting spark functions for some dataframe manipulations
from pyspark.sql.functions import *

steam_file = "steam_200k.csv"

# function for creating a dataframe using spark
def create_dataframe(_file):
    df = spark.read.options(delimiter =",").csv(f"/FileStore/tables/{_file}")
    return df

# calling function for dataframe creation with our csv file as argument
steam_dataframe = create_dataframe(steam_file)
steam_dataframe.show()

+---------+--------------------+--------+----+
|      _c0|                 _c1|     _c2| _c3|
+---------+--------------------+--------+----+
|151603712|The Elder Scrolls...|purchase|   1|
|151603712|The Elder Scrolls...|    play| 273|
|151603712|           Fallout 4|purchase|   1|
|151603712|           Fallout 4|    play|  87|
|151603712|               Spore|purchase|   1|
|151603712|               Spore|    play|14.9|
|151603712|   Fallout New Vegas|purchase|   1|
|151603712|   Fallout New Vegas|    play|12.1|
|151603712|       Left 4 Dead 2|purchase|   1|
|151603712|       Left 4 Dead 2|    play| 8.9|
|151603712|            HuniePop|purchase|   1|
|151603712|            HuniePop|    play| 8.5|
|151603712|       Path of Exile|purchase|   1|
|151603712|       Path of Exile|    play| 8.1|
|151603712|         Poly Bridge|purchase|   1|
|151603712|         Poly Bridge|    play| 7.5|
|151603712|         Left 4 Dead|purchase|   1|
|151603712|         Left 4 Dead|    play| 3.3|
|151603712|  

In [0]:
steam_dataframe.describe()

DataFrame[summary: string, _c0: string, _c1: string, _c2: string, _c3: string]

In [0]:
# Function to alter schema of the dataframe
def alter_schema_name(dataframe):
    # Renaming the Shema
    df_renamed = dataframe.withColumnRenamed("_c0", "User ID").withColumnRenamed("_c1", "Game Title").withColumnRenamed("_c2", "Member Behaviour").withColumnRenamed("_c3", "Game Play Time")
    
    #Altering the datatypes of dataframe schema
    df = df_renamed.withColumn("User ID", df_renamed["User ID"].cast("int")).withColumn("Game Play Time", df_renamed["Game Play Time"].cast("float"))
    return df

steam_dataframe = alter_schema_name(steam_dataframe)
steam_dataframe.show(20)

+---------+--------------------+----------------+--------------+
|  User ID|          Game Title|Member Behaviour|Game Play Time|
+---------+--------------------+----------------+--------------+
|151603712|The Elder Scrolls...|        purchase|           1.0|
|151603712|The Elder Scrolls...|            play|         273.0|
|151603712|           Fallout 4|        purchase|           1.0|
|151603712|           Fallout 4|            play|          87.0|
|151603712|               Spore|        purchase|           1.0|
|151603712|               Spore|            play|          14.9|
|151603712|   Fallout New Vegas|        purchase|           1.0|
|151603712|   Fallout New Vegas|            play|          12.1|
|151603712|       Left 4 Dead 2|        purchase|           1.0|
|151603712|       Left 4 Dead 2|            play|           8.9|
|151603712|            HuniePop|        purchase|           1.0|
|151603712|            HuniePop|            play|           8.5|
|151603712|       Path of

# EXPLORATORY ANALYSIS

In [0]:
#Analysis 1: Number of distinct records

steam_dataframe.distinct().count()

199293

In [0]:
#Analysis 2: Unique users

steam_dataframe.select("User ID").distinct().count()

12393

In [0]:
# Analysis 3: Number of Unique Games

steam_dataframe.select("Game Title").distinct().count()

5155

In [0]:
# #Analysis 4: Top 10 Most popularly played games

steam_dataframe.select("Game Title", "Member Behaviour", "Game Play Time").filter(steam_dataframe["Member Behaviour"] == "play").groupBy("Game Title").count().orderBy('count', ascending=False).limit(10).display()

Game Title,count
Dota 2,4841
Team Fortress 2,2323
Counter-Strike Global Offensive,1377
Unturned,1069
Left 4 Dead 2,801
Counter-Strike Source,715
The Elder Scrolls V Skyrim,677
Garry's Mod,666
Counter-Strike,568
Sid Meier's Civilization V,554


Databricks visualization. Run in Databricks to view.

In [0]:
#Analysis 5: Top 10 Games with the highest number of play time in hours

steam_dataframe.select("Game Title", "Member Behaviour", "Game Play Time").filter(steam_dataframe["Member Behaviour"] == "play").groupBy("Game Title").sum("Game Play Time").orderBy('sum(Game Play Time)', ascending=False).limit(10).display()

Game Title,sum(Game Play Time)
Dota 2,981684.6000046805
Counter-Strike Global Offensive,322771.60000587255
Team Fortress 2,173673.30000534654
Counter-Strike,134261.1000032574
Sid Meier's Civilization V,99821.30000032485
Counter-Strike Source,96075.4999980852
The Elder Scrolls V Skyrim,70889.30000342429
Garry's Mod,49725.300001084805
Call of Duty Modern Warfare 2 - Multiplayer,42009.8999973014
Left 4 Dead 2,33596.70000024885


Databricks visualization. Run in Databricks to view.

In [0]:
# Analysis 6: Top 10 Most popularly purchased games

steam_dataframe.select("Game Title", "Member Behaviour", "Game Play Time").filter(steam_dataframe["Member Behaviour"] == "purchase").groupBy("Game Title").count().orderBy('count', ascending=False).limit(10).display()

Game Title,count
Dota 2,4841
Team Fortress 2,2323
Unturned,1563
Counter-Strike Global Offensive,1412
Half-Life 2 Lost Coast,981
Counter-Strike Source,978
Left 4 Dead 2,951
Counter-Strike,856
Warframe,847
Half-Life 2 Deathmatch,823


Databricks visualization. Run in Databricks to view.

# DATA PREPARATION FOR ALS

In [0]:
# Preparing data for ML training

# Generating a uniqe ID for each game in the dataset
game_id_dataframe = steam_dataframe.select("Game Title").distinct().withColumn('Game ID', monotonically_increasing_id()).withColumnRenamed("Game Title", "Game")

game_id_dataframe.show(truncate=False)

+-------------------------------------------+-------+
|Game                                       |Game ID|
+-------------------------------------------+-------+
|Dota 2                                     |0      |
|METAL GEAR SOLID V THE PHANTOM PAIN        |1      |
|LEGO Batman The Videogame                  |2      |
|RIFT                                       |3      |
|Anodyne                                    |4      |
|Legend of Grimrock                         |5      |
|Divinity Original Sin                      |6      |
|Meltdown                                   |7      |
|SanctuaryRPG Black Edition                 |8      |
|Snuggle Truck                              |9      |
|Lunar Flight                               |10     |
|Dungeons 2                                 |11     |
|Zuma's Revenge                             |12     |
|HassleHeart                                |13     |
|Ihf Handball Challenge 12                  |14     |
|NEON STRUCT Soundtrack & Ar

In [0]:
# Joining the Generated Game ID dataframe with the main Dafaframe 

steam_dataframe_id = game_id_dataframe.join(steam_dataframe, game_id_dataframe["Game"] == steam_dataframe["Game Title"]).drop("Game").select("User ID", "Game ID", "Game Title", "Member Behaviour", "Game Play Time")
steam_dataframe_id.show()

+---------+-------+--------------------+----------------+--------------+
|  User ID|Game ID|          Game Title|Member Behaviour|Game Play Time|
+---------+-------+--------------------+----------------+--------------+
|151603712|   2609|The Elder Scrolls...|        purchase|           1.0|
|151603712|   2609|The Elder Scrolls...|            play|         273.0|
|151603712|    410|           Fallout 4|        purchase|           1.0|
|151603712|    410|           Fallout 4|            play|          87.0|
|151603712|   3868|               Spore|        purchase|           1.0|
|151603712|   3868|               Spore|            play|          14.9|
|151603712|   3820|   Fallout New Vegas|        purchase|           1.0|
|151603712|   3820|   Fallout New Vegas|            play|          12.1|
|151603712|     69|       Left 4 Dead 2|        purchase|           1.0|
|151603712|     69|       Left 4 Dead 2|            play|           8.9|
|151603712|   3340|            HuniePop|        pur

In [0]:
# Splitting data into trainning and test set

steam_dataframe_id = steam_dataframe_id.filter(steam_dataframe_id["Member Behaviour"] == "play")
trainning_set, test_set = steam_dataframe_id.randomSplit([0.8, 0.2])
trainning_set.show(truncate=False)

+-------+-------+-------------------------------------------+----------------+--------------+
|User ID|Game ID|Game Title                                 |Member Behaviour|Game Play Time|
+-------+-------+-------------------------------------------+----------------+--------------+
|5250   |0      |Dota 2                                     |play            |0.2           |
|5250   |779    |Deus Ex Human Revolution                   |play            |62.0          |
|5250   |3369   |Alien Swarm                                |play            |4.9           |
|5250   |3458   |Cities Skylines                            |play            |144.0         |
|5250   |3893   |Team Fortress 2                            |play            |0.8           |
|76767  |17     |Call of Duty Modern Warfare 2 - Multiplayer|play            |165.0         |
|76767  |44     |Counter-Strike Source                      |play            |25.0          |
|76767  |487    |Rise of Nations Extended Edition           

# CREATING ALS OBJECT INSTANCE AND TRAINNING WITH TRAINNING SET

In [0]:
# Training a model using Alternating least squares(ALS) with the trainning set
from pyspark.ml.recommendation import ALS
from pyspark.ml.evaluation import RegressionEvaluator

# Creating a class for recommender system trainning
class GameRecommenderModel: 
    def __init__(self, rank, maxIter, regParam): 
        # Setting ALS object parameters
        self.rank = rank 
        self.maxIter = maxIter 
        self.regParam = regParam 
        self.userCol = "User ID" 
        self.ratingCol = "Game Play Time" 
        self.itemCol = "Game ID" 
        self.testSet = test_set 
        self.trainningSet = trainning_set 

        # ALS object instance
        self.alsObject = ALS(rank=self.rank, maxIter=self.maxIter, regParam=self.regParam, userCol=self.userCol, ratingCol=self.ratingCol, itemCol=self.itemCol, coldStartStrategy="drop", nonnegative=True)

        # RegressionEvaluator object instance
        self.evaluator = RegressionEvaluator(metricName="rmse", labelCol=self.ratingCol, predictionCol="prediction")

    def trainModel(self):  # Model fitting method
        return self.alsObject.fit(self.trainningSet)
    
    def getTestSetPredictions(self, model): # Method to get predictions of test
        return model.transform(self.testSet)

    def getRmse(self, predictions): # Method to get RMSE of the model
        return self.evaluator.evaluate(predictions)

In [0]:
rank, maxIter, regParam = (15, 15, 0.2) # Defining values for ALS estimator hyperparameters

gameModel = GameRecommenderModel(rank, maxIter, regParam) # Creating object (ALS instance) with hyper parameters
trainnedGameModel = gameModel.trainModel() # Calling method for trainning model

2024/05/02 04:05:40 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'f6e62f4dc9b94f7e8af627421047830b', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current pyspark.ml workflow
2024/05/02 04:05:54 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/databricks/python/lib/python3.11/site-packages/mlflow/types/utils.py:393: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Valu

In [ ]:
#  Calling the getTestSetPredictions method to view predictions made from trainned model
predictions = gameModel.getTestSetPredictions(trainnedGameModel)

# EVALUATING THE INITIAL MODEL USING RMSE

In [0]:
# Viewing Root mean squared error for gameModel
print(f'Rmse for gameModel: {gameModel.getRmse(predictions)}')

Rmse for gameModel: 234.12696611401253


# RUNNING MULTIPLE EXPERIMENTAL MODELS USING PARAMGRIDBUILDER

In [0]:
from pyspark.ml.tuning import ParamGridBuilder, TrainValidationSplit


ranks, maxIters, regParams = ([15, 20, 30], [10, 20, 30], [0.05, 0.05, 0.5])

# creating the parameter grid
parameters = ParamGridBuilder().addGrid(gameModel.alsObject.rank, ranks).addGrid(gameModel.alsObject.maxIter, maxIters).addGrid(gameModel.alsObject.regParam, regParams).build()

tvs = TrainValidationSplit(estimator=gameModel.alsObject, estimatorParamMaps=parameters, evaluator=gameModel.evaluator, trainRatio=0.9)

gridSearchModel = tvs.fit(trainning_set)

2024/05/02 04:07:57 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'eeb7febe122048aaab05eb91a65cf635', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current pyspark.ml workflow
2024/05/02 05:14:03 WARNING mlflow.pyspark.ml: Model TrainValidationSplitModel_6d8034f5cf75 will not be autologged because it is not allowlisted or or because one or more of its nested models are not allowlisted. Call mlflow.spark.log_model() to explicitly log the model, or specify a custom allowlist via the spark.mlflow.pysparkml.autolog.logModelAllowlistFile Spark conf (see mlflow.pyspark.ml.autolog docs for more info).


In [0]:
bestModel = gridSearchModel.bestModel
predictions = bestModel.transform(test_set) 
predictions.limit(100).display()

User ID,Game ID,Game Title,Member Behaviour,Game Play Time,prediction
5250,1017,Portal 2,play,13.6,25.60735
76767,1017,Portal 2,play,15.0,13.176457
76767,1477,Total War ATTILA,play,207.0,217.475
76767,1635,Call of Duty Modern Warfare 3,play,15.9,21.92424
76767,3850,Call of Duty Modern Warfare 2,play,65.0,5.2042675
76767,4068,Age of Empires II HD Edition,play,13.1,21.001371
298950,0,Dota 2,play,0.5,39.138126
298950,47,Alpha Protocol,play,0.7,10.885596
298950,69,Left 4 Dead 2,play,16.3,54.101646
298950,224,LEGO MARVEL Super Heroes,play,10.8,1094.1094


# GETTING BEST HYPER PARAMETERS

In [0]:
bestRank, bestRegParam, bestMaxIter = (bestModel.rank, bestModel._java_obj.parent().getRegParam(), bestModel._java_obj.parent().getMaxIter())
print('Hyper parameters for best model: ')
print(f'Rank Parameter: {bestRank}')
print(f'regParam Parameter: {bestRegParam}')
print(f'maxIter: {bestMaxIter}')

Hyper parameters for best model: 
Rank Parameter: 30
regParam Parameter: 0.5
maxIter: 20


# TRAINNING A NEW MODEL USING BEST HYPER PARAMETERS

In [0]:
# Using best hyperparameter values to train our model

BestgameModel = GameRecommenderModel(bestRank, bestMaxIter, bestRegParam) # Using best hyperparameters to create model object
trainnedBestGameModel = BestgameModel.trainModel() 
predictions = BestgameModel.getTestSetPredictions(trainnedBestGameModel)
bestModelRmse = BestgameModel.getRmse(predictions)
print(f"Rmse for best model: {bestModelRmse}")

2024/05/02 05:14:20 INFO mlflow.utils.autologging_utils: Created MLflow autologging run with ID 'f5635f99bffe49eaaa826a7fee5e0d87', which will track hyperparameters, performance metrics, model artifacts, and lineage information for the current pyspark.ml workflow
2024/05/02 05:14:21 WARNING mlflow.utils.autologging_utils: MLflow autologging encountered a warning: "/databricks/python/lib/python3.11/site-packages/mlflow/types/utils.py:393: UserWarning: Hint: Inferred schema contains integer column(s). Integer columns in Python cannot represent missing values. If your input data contains missing values at inference time, it will be encoded as floats and will cause a schema enforcement error. The best way to avoid this problem is to infer the model schema based on a realistic data sample (training dataset) that includes missing values. Alternatively, you can declare integer columns as doubles (float64) whenever these columns may have missing values. See `Handling Integers With Missing Valu

Rmse for best model: 202.39287852427603


# GETTING TOP 10 RECOMMENDATIONS FOR USERS

In [ ]:
userRecommendations = trainnedBestGameModel.recommendForAllUsers(10)

In [0]:
#Final top 10 Recommendations for 1 user
singleUserRecommendation = userRecommendations.filter(userRecommendations["User ID"] == 26333936).select("recommendations").withColumn("recommendations", explode("recommendations")).select("recommendations.Game ID", "recommendations.rating").join(game_id_dataframe, ["Game ID"])

singleUserRecommendation.display()

Game ID,rating,Game
3697,290.24683,FINAL FANTASY XIV A Realm Reborn
4095,179.10896,"Warhammer 40,000 Dawn of War II Retribution"
998,134.50024,Battlefield Bad Company 2
4921,119.66581,Eastside Hockey Manager
3176,115.98405,Marvel Puzzle Quest
1307,107.53055,FIFA Manager 09
631,102.101,Age of Chivalry
2202,99.130585,Football Manager 2010
3772,89.587746,Football Manager 2012
817,87.74877,Primal Carnage
